# 20 SFT pilot: export the training data, measure speed and memory, smoke-test the whole path

1. **Export** (once): training data for the four SFT methods from the quality-checked pool `pool_v1`.
2. **Pilot**: six optimizer steps of real training on this GPU, starting with the step that holds the longest sequence, so the memory peak shows at once. Questions of one database share one copy of the schema (groups of 8, two groups per step). Prints tokens per second, peak memory and the projected hours and compute units for one epoch of one method.
3. **Smoke run**: three training steps, then predictions for 20 validation and 20 dev questions with that adapter, to prove vLLM loads our adapters before any full run.

Run it on an **A100** runtime (an L4's 24 GB is too small for the longest sequences). About 20 to 30 minutes; the export is skipped when it already exists.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
POOL = "pool_v1"
EXPORT = "sft_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1. Export (CPU work, a few minutes; skipped when it already exists)

In [ ]:
if not os.path.exists(f"{R}/{EXPORT}/export_meta.json"):
    !python -m effsql.train export --pool-run {POOL} --run {EXPORT} --arms gold,best,weighted,tagged
else:
    print("export exists:", EXPORT)

### 2. Pilot: speed, memory, projected cost of one epoch

In [ ]:
!python -m effsql.train pilot --export {EXPORT} --arm weighted --steps 6

### 3. Smoke run: train 3 steps, predict 20 validation and 20 dev questions with the adapter

In [ ]:
SMOKE = f"{EXPORT}_smoke_groups"  # a new name: the first smoke run predates the schema groups
!python -m effsql.train sft --export {EXPORT} --arm tagged --run {SMOKE} --max-steps 3
!python -m effsql.train predict --run {SMOKE} --splits val,dev --limit 20 --allow-incomplete

### Done: disconnect now

*Runtime > Disconnect and delete runtime.* Share the pilot output: it confirms the cost of the four training notebooks.